In [2]:
import numpy as np
import math
import csv

In [3]:
def read_data(tennisdata):
    with open(tennisdata, 'r') as csvfile:
        datareader = csv.reader(csvfile, delimiter=',')
        headers = next(datareader)

        metadata = []
        traindata = []

        for name in headers:
            metadata.append(name)

        for row in datareader:
            traindata.append(row)

    return metadata, traindata

In [4]:
class Node:
    def __init__(self, attribute):
        self.attribute = attribute
        self.children = []
        self.answer = ""

    def __str__(self):
        return self.attribute


In [5]:
def subtables(data, col, delete):
    items = np.unique(data[:, col])
    count = np.zeros(items.shape[0], dtype=np.int32)

    for x in range(items.shape[0]):
        for y in range(data.shape[0]):
            if data[y, col] == items[x]:
                count[x] += 1

    dictionary = {}

    for x in range(items.shape[0]):
        dictionary[items[x]] = np.empty(
            (int(count[x]), data.shape[1]), dtype="|S32"
        )
        pos = 0
        for y in range(data.shape[0]):
            if data[y, col] == items[x]:
                dictionary[items[x]][pos] = data[y]
                pos += 1

        if delete:
            dictionary[items[x]] = np.delete(dictionary[items[x]], col, 1)

    return items, dictionary

In [6]:
def entropy(S):
    items = np.unique(S)

    if items.size == 1:
        return 0

    counts = np.zeros(items.shape[0])
    entropy_value = 0.0

    for x in range(items.shape[0]):
        counts[x] = np.sum(S == items[x]) / (S.size * 1.0)

    for p in counts:
        entropy_value += -p * math.log(p, 2)

    return entropy_value

In [7]:
def gain_ratio(data, col):
    items, dictionary = subtables(data, col, delete=False)
    total_size = data.shape[0]

    entropies = np.zeros(items.shape[0])
    intrinsic = np.zeros(items.shape[0])

    for x in range(items.shape[0]):
        ratio = dictionary[items[x]].shape[0] / (total_size * 1.0)
        entropies[x] = ratio * entropy(dictionary[items[x]][:, -1])
        intrinsic[x] = ratio * math.log(ratio, 2)

    total_entropy = entropy(data[:, -1])
    iv = -1 * sum(intrinsic)

    for x in range(entropies.shape[0]):
        total_entropy -= entropies[x]

    return total_entropy / iv

In [8]:
def create_node(data, metadata):
    if np.unique(data[:, -1]).shape[0] == 1:
        node = Node("")
        node.answer = np.unique(data[:, -1])[0]
        return node

    gains = np.zeros((data.shape[1] - 1, 1))

    for col in range(data.shape[1] - 1):
        gains[col] = gain_ratio(data, col)

    split = np.argmax(gains)
    node = Node(metadata[split])

    metadata = np.delete(metadata, split, 0)
    items, dictionary = subtables(data, split, delete=True)

    for x in range(items.shape[0]):
        child = create_node(dictionary[items[x]], metadata)
        node.children.append((items[x], child))

    return node

In [9]:
def empty(size):
    s = ""
    for x in range(size):
        s += " "
    return s

In [10]:
def print_tree(node, level):
    if node.answer != "":
        print(empty(level), node.answer)
        return

    print(empty(level), node.attribute)

    for value, n in node.children:
        print(empty(level + 1), value)
        print_tree(n, level + 2)

In [11]:
# ---- MAIN PROGRAM ----
metadata, traindata = read_data("../datasets/Data3.csv")
data = np.array(traindata)

node = create_node(data, metadata)
print_tree(node, 0)

 Outlook
  Overcast
   b'Yes'
  Rainy
   Windy
    b'False'
     b'Yes'
    b'True'
     b'No'
  Sunny
   Humidity
    b'High'
     b'No'
    b'Normal'
     b'Yes'
